# Task 5 - Strategy Backtesting

**Objective:** Validate the Task 4 optimal portfolio by simulating its performance on a held-out
historical window and comparing it against a passive 60% SPY / 40% BND benchmark.

**Backtesting window:** January 2025 - January 2026 (data not used to train the Task 2/3 models).

**Strategy weights (from Task 4, Maximum Sharpe Ratio portfolio):** TSLA = 0%, BND = 0%, SPY = 100%.


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from backtest import (
    compute_daily_returns,
    simulate_portfolio,
    simulate_benchmark,
    performance_metrics,
    metrics_table,
    plot_backtest,
    run_backtest,
)

plt.style.use("seaborn-v0_8-whitegrid")


## 1. Load Price Data and Define the Backtesting Window

Use the same TSLA / BND / SPY adjusted-close price data pulled in Task 1. Slice out the
**last year** of the dataset (2025-01-01 to 2026-01-01) — this window must not overlap with
whatever period was used to train the Task 2/3 forecasting models.


In [ ]:
# --- Load your actual Task 1 price data here ---
# Example, if you saved cleaned prices to a CSV in Task 1:
# prices = pd.read_csv("../data/processed/adj_close_prices.csv", index_col=0, parse_dates=True)

prices = pd.read_csv("../data/processed/adj_close_prices.csv", index_col=0, parse_dates=True)
prices = prices[["TSLA", "BND", "SPY"]].sort_index()

BACKTEST_START = "2025-01-01"
BACKTEST_END = "2026-01-01"

backtest_prices = prices.loc[BACKTEST_START:BACKTEST_END]
print(f"Backtest window: {backtest_prices.index.min().date()} -> {backtest_prices.index.max().date()}")
print(f"Trading days: {len(backtest_prices)}")
backtest_prices.head()


## 2. Define Strategy and Benchmark Weights

- **Strategy:** the Task 4 Maximum Sharpe Ratio portfolio (TSLA 0% / BND 0% / SPY 100%).
- **Benchmark:** a static, passive 60% SPY / 40% BND portfolio, rebalanced monthly.

Both are simulated with **monthly rebalancing** (the "advanced" option from the task
instructions) so the comparison reflects how each would actually have been managed, not just
a frozen buy-and-hold snapshot. Swap `rebalance_freq="M"` for `None` below to see the simpler
buy-and-hold variant instead.


In [ ]:
strategy_weights = {"SPY": 1.00}       # Task 4 Max Sharpe portfolio (TSLA and BND both 0%)
benchmark_weights = {"SPY": 0.60, "BND": 0.40}

REBALANCE_FREQ = "M"   # "M" = monthly rebalancing; use None for pure buy-and-hold
RISK_FREE_RATE = 0.04  # same assumption used in Task 4


## 3. Simulate the Strategy and the Benchmark


In [ ]:
results = run_backtest(
    backtest_prices,
    strategy_weights=strategy_weights,
    benchmark_weights=benchmark_weights,
    rebalance_freq=REBALANCE_FREQ,
    risk_free_rate=RISK_FREE_RATE,
)

strategy_cum = results["strategy_cum"]
benchmark_cum = results["benchmark_cum"]
strategy_metrics = results["strategy_metrics"]
benchmark_metrics = results["benchmark_metrics"]
table = results["table"]


## 4. Cumulative Returns: Strategy vs. Benchmark


In [ ]:
fig = plot_backtest(
    strategy_cum,
    benchmark_cum,
    title="Task 4 Strategy (100% SPY) vs. 60/40 Benchmark - Backtest",
)
plt.show()


## 5. Performance Metrics


In [ ]:
table_display = table.copy()
for col in table_display.columns:
    table_display[col] = table_display[col].map(lambda x: f"{x:.2%}" if abs(x) < 10 else f"{x:.4f}")
# Sharpe ratio isn't a percentage - format it separately
table_display.loc["Sharpe Ratio"] = [f"{strategy_metrics['sharpe_ratio']:.4f}",
                                      f"{benchmark_metrics['sharpe_ratio']:.4f}"]
table_display


## 6. Conclude and Reflect

**Did the strategy outperform the benchmark?**

*[Fill in once you have real numbers: state total/annualized return and Sharpe ratio for
both, and say plainly whether the 100%-SPY strategy beat, matched, or lagged the 60/40
benchmark over this specific 2025-2026 window, and by how much.]*

**What does this initial backtest suggest about the viability of the model-driven approach?**

*[Note that the Task 4 recommendation reduced, in this instance, to being fully invested in
SPY - so this backtest is largely a test of "was 100% equities better than a 60/40 blend over
this one-year window," rather than a test of the forecasting model's stock-picking skill,
since the ARIMA(0,1,0) forecast contributed no directional view on TSLA. A more genuine test
of the model-driven approach would come from a period/allocation where the forecast assigns a
non-trivial weight to TSLA.]*

**What are the limitations of this backtest?**

- Single backtesting window (2025-2026): results are highly sample-specific and don't
  establish statistical significance.
- Static weights are not re-derived from a rolling re-forecast; only the *rebalancing*
  reproduces target weights, not a fresh Task 2-4 pipeline run at each rebalance date.
- No transaction costs, taxes, slippage, or bid-ask spread are modeled.
- The 4% risk-free rate is a fixed assumption, not the actual risk-free rate path over the
  window.
- Survivorship/point-in-time bias: the benchmark and universe (TSLA/BND/SPY) were chosen with
  hindsight, not selected blind to how 2025-2026 played out.
